# Superstore Sales Analysis

This notebook analyzes retail sales data to identify business insights related to revenue, profitability, regional performance, and product trends.

## 1. Objective

The objective of this analysis is to explore historical sales data and answer key business questions such as:

- Which categories generate the most revenue and profit?
- Which regions perform best?
- How do sales change over time?
- Which products contribute the most to revenue?

The final goal is to generate actionable business insights and recommendations.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load Data

In [2]:
df = pd.read_csv('../data/superstore.csv', encoding='latin-1', decimal=',')
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1.0,CA-2016-152156,8.11.2016,11.11.2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.0,0%,41.91
1,2.0,CA-2016-152156,8.11.2016,11.11.2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3.0,0%,219.58
2,3.0,CA-2016-138688,12.06.2016,16.06.2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2.0,0%,6.87
3,4.0,US-2015-108966,11.10.2015,18.10.2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5.0,45%,-383.03
4,5.0,US-2015-108966,11.10.2015,18.10.2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2.0,20%,2.52


## 3. Data Overview

In this section, we explore the structure and key characteristics of the dataset:

- Dataset dimensions (rows and columns)
- Data types and column structure
- Missing values overview

In [3]:
df.shape

(9996, 21)

### Missing Values Analysis

We check for missing values in the dataset to understand data quality issues.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9996 entries, 0 to 9995
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         9994 non-null   object 
 1   order_id       9995 non-null   object 
 2   order_date     9994 non-null   object 
 3   ship_date      9994 non-null   object 
 4   ship_mode      9994 non-null   object 
 5   customer_id    9994 non-null   object 
 6   customer_name  9994 non-null   object 
 7   segment        9994 non-null   object 
 8   country        9994 non-null   object 
 9   city           9994 non-null   object 
 10  state          9994 non-null   object 
 11  postal_code    9994 non-null   object 
 12  region         9994 non-null   object 
 13  product_id     9994 non-null   object 
 14  category       9994 non-null   object 
 15  sub_category   9994 non-null   object 
 16  product_name   9996 non-null   object 
 17  sales          9996 non-null   float64
 18  quantity

In [5]:
df.isna().sum().sort_values(ascending=False).to_frame(name='missing_count')

,missing_count
row_id,2
city,2
discount,2
quantity,2
sub_category,2
category,2
product_id,2
region,2
postal_code,2
state,2


## 4. Data Cleaning

In this step, we clean and prepare the dataset for analysis:

- Fix inconsistent numeric formats (e.g., discount, quantity)
- Convert columns to appropriate data types
- Removing Missing Values
- Ensure data consistency

The data cleaning process is performed step-by-step as follows:

### Numeric Cleaning

We clean columns with inconsistent formatting such as percentage signs and commas.
This step ensures numeric columns are free from formatting inconsistencies such as symbols and separators.

In [6]:
# 1. Numeric cleaning
df['discount'] = (
    df['discount']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

df['quantity'] = (
    df['quantity']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

### Convert Cleaned Columns to Numeric

We convert cleaned columns (discount and quantity) into numeric data types.

In [7]:
# 2. Convert to numeric
df['discount'] = pd.to_numeric(df['discount'], errors='coerce') / 100
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

### Date Conversion

We convert date columns into datetime format for time-based analysis.

In [8]:
# 3. Date conversion
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)

### Removing Missing Values

We remove rows with missing critical values such as sales, profit, and quantity.

In [9]:
# 4. Drop missing ONLY AFTER conversions
df = df.dropna(subset=['sales', 'profit', 'quantity']).copy()

### Postal Code Fix

Postal codes are converted to string format to preserve leading zeros and ensure consistency in categorical analysis.

In [10]:
# 5. Fix postal_code
df['postal_code'] = df['postal_code'].astype(str)

In [11]:
# check
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   row_id         9994 non-null   object        
 1   order_id       9994 non-null   object        
 2   order_date     9994 non-null   datetime64[ns]
 3   ship_date      9994 non-null   datetime64[ns]
 4   ship_mode      9994 non-null   object        
 5   customer_id    9994 non-null   object        
 6   customer_name  9994 non-null   object        
 7   segment        9994 non-null   object        
 8   country        9994 non-null   object        
 9   city           9994 non-null   object        
 10  state          9994 non-null   object        
 11  postal_code    9994 non-null   object        
 12  region         9994 non-null   object        
 13  product_id     9994 non-null   object        
 14  category       9994 non-null   object        
 15  sub_category   9994 non-nu

Before diving into analysis, we ensure the dataset is clean, consistent, and ready for reliable insights.

## 5. Exploratory Data Analysis (EDA)



In this section, we perform exploratory data analysis to extract key business insights related to:

- Sales performance
- Profitability
- Regional trends
- Product categories

In [12]:
# Sales summary
df[['sales', 'profit', 'quantity', 'discount']].describe()

,sales,profit,quantity,discount
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858022,28.656973,3.789574,0.156203
std,623.245131,234.260203,2.225110,0.206452
min,0.440000,-6599.980000,1.000000,0.000000
25%,17.280000,1.730000,2.000000,0.000000
50%,54.490000,8.665000,3.000000,0.200000
75%,209.940000,29.360000,5.000000,0.200000
max,22638.480000,8399.980000,14.000000,0.800000


In [13]:
# Negative profit analysis
df[df['profit'] < 0].describe()

,order_date,ship_date,sales,quantity,discount,profit
count,1871,1871,1871.000000,1871.000000,1871.000000,1871.000000
mean,2016-04-27 03:02:24.307856896,2016-05-01 02:22:23.025120256,250.511694,3.762694,0.480887,-83.448348
min,2014-01-04 00:00:00,2014-01-08 00:00:00,0.440000,1.000000,0.100000,-6599.980000
25%,2015-05-10 00:00:00,2015-05-13 12:00:00,12.500000,2.000000,0.200000,-58.660000
50%,2016-06-12 00:00:00,2016-06-17 00:00:00,71.090000,3.000000,0.400000,-18.090000
75%,2017-05-05 12:00:00,2017-05-10 00:00:00,284.920000,5.000000,0.700000,-6.265000
max,2017-12-30 00:00:00,2018-01-03 00:00:00,22638.480000,14.000000,0.800000,-0.090000
std,NaN,NaN,715.067319,2.141347,0.235080,284.423348


In [14]:
# Compare discount for profitable vs non-profitable
df.groupby(df['profit'] < 0)['discount'].mean()

profit
False    0.081417
True     0.480887
Name: discount, dtype: float64

### Key Insight: Discount vs Profitability

Analysis shows that orders with negative profit have significantly higher discounts (48%) compared to profitable orders (8%).

This suggests that excessive discounting is a major driver of losses and should be carefully optimized.

In [15]:
# Profit by region
df.groupby('region')['profit'].sum().sort_values()

region
Central     39706.45
South       46749.71
East        91522.84
West       108418.79
Name: profit, dtype: float64

### Key Insight: Regional Profitability

The Central region generates the lowest total profit compared to other regions.

This indicates potential operational inefficiencies, pricing issues, or high discounting in this region that require further investigation.

In [16]:
# Profit by category
df.groupby('category')['profit'].sum().sort_values()

category
Furniture           18451.25
Office Supplies    122490.88
Technology         145455.66
Name: profit, dtype: float64

### Key Insight: Category Profitability

The Furniture category generates significantly lower profit compared to Technology and Office Supplies.

This suggests potential issues such as high discounting, low margins, or operational inefficiencies within this category that require further analysis.

In [17]:
# Profit by sub-category
df.groupby('sub_category')['profit'].sum().sort_values()

sub_category
Tables        -17725.59
Bookcases      -3472.56
Supplies       -1188.99
Fasteners        949.53
Machines        3384.73
Labels          5546.18
Art             6527.96
Envelopes       6964.10
Furnishings    13059.25
Appliances     18138.07
Storage        21279.05
Chairs         26590.15
Binders        30221.64
Paper          34053.34
Accessories    41936.78
Phones         44516.25
Copiers        55617.90
Name: profit, dtype: float64

### Key Insight: Sub-category Profitability

The Tables sub-category is the largest contributor to losses, followed by Bookcases and Supplies.

This indicates significant pricing or discounting issues within these product groups, particularly Tables, which require immediate attention.

In [18]:
# Top loss-making products
loss_products = (
    df[df['profit'] < 0]
    .groupby('product_name')['profit']
    .sum()
    .sort_values()
    .head(10)
)

loss_products

product_name
Cubify CubeX 3D Printer Double Head Print                                     -9239.97
GBC DocuBind P400 Electric Binding System                                     -6859.39
Lexmark MX611dhe Monochrome Laser Printer                                     -5269.97
GBC Ibimaster 500 Manual ProClick Binding System                              -5098.57
GBC DocuBind TL300 Electric Binding System                                    -4162.04
Cubify CubeX 3D Printer Triple Head Print                                     -3839.99
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind   -3431.67
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases                      -3107.52
Ibico EPK-21 Electric Binding System                                          -2929.48
Bush Advantage Collection Racetrack Conference Table                          -2545.26
Name: profit, dtype: float64

### Key Insight: Top Loss-making Products (High-value items)

The most unprofitable products are primarily high-value items such as printers, binding systems, and conference tables.

This suggests that expensive products are either heavily discounted or incorrectly priced, leading to substantial losses.

This issue is particularly critical as a small number of products are responsible for a large portion of total losses.

## Final Recommendations

Based on the analysis, the following actions are recommended:

- Review and reduce discount levels on high-value products (e.g., printers, tables)
- Re-evaluate pricing strategy for loss-making products
- Consider discontinuing or replacing consistently unprofitable products
- Focus on high-performing categories such as Technology and Office Supplies
- Investigate operational inefficiencies in low-performing regions (e.g., Central)

These recommendations can help improve profitability, optimize pricing strategies, and reduce losses across key business areas.